# AI/ML Sales Forecasting
This notebook demonstrates data cleaning, EDA, feature engineering, model training, evaluation, and forecasting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv('sales_data.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.drop_duplicates().sort_values('date').reset_index(drop=True)
df.head()

## Exploratory Data Analysis

In [ ]:
print(df.info())
display(df.describe())
plt.figure(figsize=(10,5))
plt.plot(df['date'], df['sales'], marker='o')
plt.title('Monthly Sales Trend')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Feature Engineering

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df['lag_1'] = df['sales'].shift(1)
df['lag_2'] = df['sales'].shift(2)
df['rolling_mean_3'] = df['sales'].shift(1).rolling(3).mean()
df = df.dropna().reset_index(drop=True)
df.head()

## Model Training and Evaluation

In [ ]:
features = ['promotion','ad_spend','year','month','quarter','lag_1','lag_2','rolling_mean_3']
X = df[features]
y = df['sales']
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred, squared=False),
        'R2': r2_score(y_test, pred)
    })
pd.DataFrame(results).sort_values('RMSE')